# 🏎️ Notebook 07 — Driver Cards
### Formula 1 Race Prediction & Driver Analytics



## What This Notebook Does

This is the final and most visible output of the entire project.
For every driver with sufficient race history, we generate a
**Driver Performance Card** — a single-page visual profile containing:

1. **General Statistics** — races, wins, podiums, points, avg finish
2. **Season Trend** — points and avg finish position across seasons
3. **Circuit Heatmap** — best and worst circuits
4. **Reliability** — DNF rate and finish rate
5. **Strength Labels** — automatically assigned performance tags

---

## Why Driver Cards Matter

All the previous notebooks built the analytical infrastructure:
- Notebook 01–02: collected and cleaned data
- Notebook 03: engineered 19 predictive features
- Notebook 04: validated patterns with visualizations
- Notebook 05–06: trained and evaluated ML models

Driver Cards are the **human-readable output** of that infrastructure.
They answer the question every F1 fan asks:
*"Who is this driver, really — across circuits, across seasons?"*


## 🎨 Driver Card Design

Each card is a single 16×10 inch figure divided into 5 panels:

```
┌─────────────────────────────────────────────────────────┐
│  DRIVER NAME          │  GENERAL STATS (races/wins/pts) │
│  Constructor  Season  │  Avg finish / Avg grid / DNF%   │
├───────────────────────┴─────────────────────────────────┤
│  SEASON TREND                │  CIRCUIT HEATMAP         │
│  Points per season (line)    │  Avg finish by circuit   │
│  Avg finish per season (line)│  (top 8 circuits)        │
├──────────────────────────────┴──────────────────────────┤
│  STRENGTH LABELS             │  RELIABILITY PIE CHART   │
│  🏆 Race Winner              │  Finish vs DNF           │
│  ⚡ Strong Qualifier         │                          │
│  🔄 Consistent Scorer        │                          │
└─────────────────────────────────────────────────────────┘
```

**Color scheme:**
- Background: dark (#15151E)
- Accent: F1 Red (#E8002D)
- Text: white / silver
- Good performance: green (#27AE60)
- Weak performance: red (#E8002D)


# Import Libs

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec 
import matplotlib.patches as mpatches 
from matplotlib.patches import FancyBboxPatch
import seaborn as sns 
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
#Color Palette
F1_RED = '#E8002D'
F1_DARK = '#15151E'
F1_SILVER = '#C0C0C0'
F1_WHITE = '#FFFFFF'
F1_GOLD = '#FFD700'
F1_GREEN = '#27AE60'
F1_BLUE = '#3498DB'


#Constructor brands colors
TEAM_COLORS = {
    'mercedes':     '#00D2BE',
    'red_bull':     '#3671C6',
    'ferrari':      '#DC0000',
    'mclaren':      '#FF8000',
    'alpine':       '#0090FF',
    'aston_martin': '#006F62',
    'williams':     '#64C4FF',
    'alphatauri':   '#2B4562',
    'alfa':         '#900000',
    'haas':         '#B6BABD',
    'renault':      '#FFF500',
    'racing_point': '#F596C8',

}

#output dir
os.makedirs('../output/driver_cards', exist_ok=True)

print('Imports complete')
print('Output dir: ../output/driver_cards/')

Imports complete
Output dir: ../output/driver_cards/


# Load Data

In [4]:
#full feature dataset (2018-2026)
df = pd.read_csv('../data/processed/f1_features_2018_2026.csv')

print('Main dataset loaded')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Seasons: {df.season.min()} - {df.season.max()}')
print(f'Drivers: {df.driver_name.nunique()}')


Main dataset loaded
Shape: 3,458 rows x 48 columns
Seasons: 2018 - 2025
Drivers: 43


# Driver Eligibility Filter

In [5]:
MIN_RACES = 20

driver_race_counts = (
    df.groupby('driver_name')
    .size()
    .reset_index(name='total_races')
    .sort_values('total_races', ascending=False)
)

eligible_drivers = (
    driver_race_counts[driver_race_counts.total_races >= MIN_RACES]
    ['driver_name']
    .tolist()
)

print(f'Minimum races required: {MIN_RACES}')
print(f'Eligible drivers: {len(eligible_drivers)}')
print(f'Excluded drivers:   '
      f'{len(driver_race_counts) - len(eligible_drivers)}'
      f'(fewer than {MIN_RACES} starts)')

print('\nEligible drivers:')
for i, name in enumerate(eligible_drivers, 1):
    count = driver_race_counts[
        driver_race_counts.driver_name == name
    ]['total_races'].values[0]
    print(f'{i:2d}. {name:<30} ({count} races)')

Minimum races required: 20
Eligible drivers: 39
Excluded drivers:   4(fewer than 20 starts)

Eligible drivers:
 1. Charles Leclerc                (173 races)
 2. Pierre Gasly                   (173 races)
 3. Max Verstappen                 (173 races)
 4. Carlos Sainz                   (172 races)
 5. Lewis Hamilton                 (172 races)
 6. Lance Stroll                   (171 races)
 7. Lando Norris                   (152 races)
 8. George Russell                 (152 races)
 9. Esteban Ocon                   (151 races)
10. Valtteri Bottas                (149 races)
11. Sergio Pérez                   (147 races)
12. Fernando Alonso                (135 races)
13. Alexander Albon                (129 races)
14. Daniel Ricciardo               (128 races)
15. Kevin Magnussen                (125 races)
16. Nico Hülkenberg                (117 races)
17. Yuki Tsunoda                   (114 races)
18. Sebastian Vettel               (101 races)
19. Kimi Räikkönen                 (79 race

# Section 1 - Compute Driver Statistics

**Goal:** Pre-compute all statistics needed for Driver Cards
so the card generation loop is fast and clean.

We compute three types of statistics:

**1. Career-level stats** (one row per driver):
- Total races, wins, podiums, points
- Average finishing position, average qualifying position
- DNF rate, finish rate
- Best ever finish, best ever qualifying

**2. Season-level stats** (one row per driver per season):
- Total points per season
- Average finish per season
- Wins and podiums per season
- Used for the trend chart on the card

**3. Circuit-level stats** (one row per driver per circuit):
- Average finish at each circuit
- Average points at each circuit
- Number of visits
- Used for the circuit heatmap on the card


## Career Stats

In [6]:
career_stats = (
    df.groupby('driver_name')
    .agg(
        total_races = ('position', 'count'),
        total_wins = ('position', lambda x: (x==1).sum()),
        total_podiums = ('podium', 'sum'),
        total_points = ('points', 'sum'),
        avg_finish = ('position', 'mean'),
        avg_grid = ('grid', 'mean'),
        dnf_count = ('dnf', 'sum'),
        best_finish = ('position', 'min'),
        worst_finish = ('position', 'max'),
        total_seasons = ('season', 'nunique'),
        first_season = ('season', 'min'),
        last_season = ('season', 'max'),
    )
    .reset_index()
)


career_stats['dnf_rate'] = career_stats['dnf_count'] / career_stats['total_races']
career_stats['finish_rate'] = 1 - career_stats['dnf_count']
career_stats['win_rate'] = career_stats['total_wins'] / career_stats['total_races']
career_stats['podium_rate'] = career_stats['total_podiums'] / career_stats['total_races']
career_stats['points_per_race'] = (
    career_stats['total_points'] / career_stats['total_races']
)

#last know constructor for each driver
last_constructor = (
    df.sort_values('season')
    .groupby('driver_name')['constructor_id']
    .last()
    .reset_index()
    .rename(columns={'constructor_id': 'last_constructor'})
)

career_stats = career_stats.merge(last_constructor, on = 'driver_name')

print(f'Career stats computed for {len(career_stats)} drivers\n')
print(career_stats[career_stats.driver_name.isin(eligible_drivers[:3])][
    ['driver_name', 'total_races', 'total_wins', 'total_podiums',
     'total_points', 'avg_finish', 'dnf_rate']
].to_string(index=False))


Career stats computed for 43 drivers

    driver_name  total_races  total_wins  total_podiums  total_points  avg_finish  dnf_rate
Charles Leclerc          173           8             50      1588.000       7.445     0.162
 Max Verstappen          173          68            116      2880.500       4.191     0.104
   Pierre Gasly          173           1              5       446.000      11.653     0.272


## Season Stats

In [7]:
season_stats = (
    df.groupby(['driver_name', 'season'])
    .agg(
        season_points = ('points', 'sum'),
        season_wins = ('position', lambda x: (x==1).sum()),
        season_podiums = ('podium', 'sum'),
        season_races = ('position', 'count'),
        season_dnfs = ('dnf', 'sum'),
        avg_finish = ('position', 'mean'),
        avg_grid = ('grid', 'mean'),
    )
    .reset_index()
)

season_stats['season_dnf_rate'] = (
    season_stats['season_dnfs'] / season_stats['season_races']
)

print(f'Season stats computed: {len(season_stats)} driver-season combinations')


Season stats computed: 173 driver-season combinations


## Circuit Stats

In [8]:
circuit_stats = (
    df.groupby(['driver_name', 'circuit_id'])
    .agg(
        circuit_races = ('position', 'count'),
        avg_finish = ('position', 'mean'),
        avg_points = ('points', 'mean'),
        best_finish = ('position', 'min'),
        circuit_dnf_rate = ('dnf', 'mean'),
        circuit_wins = ('position', lambda x: (x==1).sum()),
        circuit_podiums = ('podium', 'sum'),
    )
    .reset_index()
)

#only include driver-circuit combinations with 2+ visits
circuit_stats = circuit_stats[circuit_stats.circuit_races >= 2].copy()

#track strength score (high = better)
circuit_stats['track_strength'] = 21.0 - circuit_stats['avg_finish']

print(f'Circuit stats computed: {len(circuit_stats)} driver-circuit combinations'
      f'(min 2 visits)')

Circuit stats computed: 691 driver-circuit combinations(min 2 visits)
